In [25]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pyprojroot import here

In [26]:
INTERIM_DATA_DIR = here() / "data" / "interim"
RESULTS_DATA_DIR = here() / "results" / "esda"
RAW_DATA_DIR = here() / "data" / "raw"

In [27]:
adj = pd.read_csv(INTERIM_DATA_DIR/"adj_combined.csv", index_col=0)
df = pd.read_csv(INTERIM_DATA_DIR/"population_2015_2025.csv")
gdf = gpd.read_file(RAW_DATA_DIR/"geographic/municipalities/00mun.shp")

adj.index   = adj.index.astype(str).str.zfill(5)
adj.columns = adj.columns.astype(str).str.zfill(5)

In [28]:
gdf_projected = gdf.to_crs(epsg=6933)
gdf_projected['area_sq_meters'] = gdf_projected.geometry.area
gdf_projected['area_sq_km'] = gdf_projected['area_sq_meters'] / 1_000_000

In [29]:
df = df.rename(columns={"cve_mun": "CVEGEO", "anio": "year"}) 
df['CVEGEO'] = df['CVEGEO'].astype(str).str.zfill(5)
df = df[df["year"] == 2025].copy().reset_index(drop=True)
df.drop(columns="Unnamed: 0", inplace=True)

In [30]:
df_density.columns

Index(['year', 'population', 'NOMGEO', 'area_sq_km', 'density'], dtype='object')

In [31]:
df_density = df.merge(gdf_projected, on="CVEGEO", how='right')
df_density.drop(columns=["CVE_ENT", "CVE_MUN", "geometry", "area_sq_meters"], inplace=True)
df_density['density'] = df_density['population'] / df_density['area_sq_km']
df_density.to_csv(INTERIM_DATA_DIR/"population_density.csv")

In [32]:
df_density["CVEGEO"] = df_density["CVEGEO"].astype(str).str.zfill(5)
df_density = df_density.set_index("CVEGEO")

In [33]:
median_density = df_density["density"].median()          # ≈ 113.9 hab/km²
df_density["density"] = df_density["density"].fillna(median_density)
print(f"Median density used for imputation: {median_density:.2f} hab/km²")
print(f"NaNs remaining: {df_density['density'].isna().sum()}")   # should be 0

Median density used for imputation: 113.91 hab/km²
NaNs remaining: 0


In [34]:
cvegeos = adj.index.tolist()
density = df_density.loc[cvegeos, "density"].values

In [35]:
A = adj.values.astype(float)
W_raw = A * density[np.newaxis, :]       # W_raw[i,j] = A[i,j] * D[j]

row_sums = W_raw.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1              # guard for islands
W = W_raw / row_sums

In [36]:
print(f"Matrix shape       : {W.shape}")
print(f"NaNs in W          : {np.isnan(W).sum()}")
print(f"Row sums min/max   : {W.sum(axis=1).min():.6f} / {W.sum(axis=1).max():.6f}")
print(f"Islands (row sum=0): {(W.sum(axis=1) == 0).sum()}")
print(f"Non-zero entries   : {(W > 0).sum()}")

Matrix shape       : (2478, 2478)
NaNs in W          : 0
Row sums min/max   : 0.000000 / 1.000000
Islands (row sum=0): 1
Non-zero entries   : 13528


In [37]:
W_df = pd.DataFrame(W, index=cvegeos, columns=cvegeos)
W_df.to_csv(INTERIM_DATA_DIR/"W_density_contiguity.csv")
print("Saved → W_density_contiguity.csv")

Saved → W_density_contiguity.csv
